# ETL del RNPDNO, autónomo y explicado

Este cuaderno contiene las funciones y transformaciones del ETL en sus propias celdas: no importa `config.py` ni módulos de `src/` y solo procesa archivos **locales**.

Cada etapa muestra métricas y vistas previas para observar cómo cambia el DataFrame. Por privacidad, nombres, CURP y FUB aparecen ocultos por defecto. La exportación está apagada por defecto.

**Orden sugerido:** ejecuta las celdas de arriba hacia abajo; primero revisa las rutas de la configuración.

In [ ]:
# 1. PREPARACIÓN: librerías, rutas y parámetros editables
from pathlib import Path
from datetime import date
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

# Raíz: supone que este cuaderno vive dentro de la carpeta notebooks/ del proyecto.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Edita aquí las rutas si usas otro corte o cambiaste archivos auxiliares.
INPUT_RNPDNO = PROJECT_ROOT / "input" / "rnpdno" / "reporte_personas_completo_16102026.parquet"
RENAPO_FILES = [
    PROJECT_ROOT / "input" / "cruces_curp" / "CURP_act260525.csv",
    PROJECT_ROOT / "input" / "cruces_curp" / "curps_reconstruidas_210625.xlsx",
    PROJECT_ROOT / "input" / "cruces_curp" / "curps_con_resultado_prueba_01072025.xlsx",
    PROJECT_ROOT / "input" / "cruces_curp" / "curps_con_resultado_prueba_050825.xlsx",
]
OUTPUT_FILE = PROJECT_ROOT / "output" / "Resultado_1_Procesamiento_ETL_RNPDNO_autonomo.parquet"

# Privacidad: True muestra datos de identificación en las vistas previas; déjalo False.
MOSTRAR_DATOS_SENSIBLES = False
# Opcional: ID numérico INEGI para procesar solo una entidad. None = toda la base.
ID_ESTADO_FILTRO = None
# Dejar apagado hasta revisar el resultado en memoria.
EXPORTAR_RESULTADO = False

HOJAS = {
    "PENDIENTES": "ACTUALIZACION_PENDIENTE",
    "LOCALIZADAS": "LOCALIZADAS",
    "DESAPARECIDAS": "DESAPARECIDAS",
}
ESTATUS_LOCALIZADAS = {"LOCALIZADA CON VIDA", "LOCALIZADA SIN VIDA"}
CODIGOS_ESTADO_CURP = {
    "AS", "BC", "BS", "CC", "CL", "CM", "CS", "CH", "DF", "DG", "GT", "GR",
    "HG", "JC", "MC", "MN", "MS", "NT", "NL", "OC", "PL", "QT", "QR", "SP",
    "SL", "SR", "TC", "TS", "TL", "VZ", "YN", "ZS", "NE",
}

# Columna estándar -> alias posibles en los archivos de origen.
COLUMN_MAPPING = {
    "fub": ["fub", "folio_fub", "folio_rnpdno", "folio"],
    "nombre_vic": ["nombre_vic", "nombre"],
    "primerapellido_vic": ["primerapellido_vic", "primer_apellido"],
    "segundoapellido_vic": ["segundoapellido_vic", "segundo_apellido"],
    "curp_vic": ["curp_vic", "curp", "curp_victima"],
    "sexo": ["sexo"],
    "fechanacimiento": ["fechanacimiento", "fecha_nacimiento", "fha_nac"],
    "idestadonacimiento_vic": ["idestadonacimiento_vic"],
    "idestado": ["idestado", "cve_ent", "estado_id", "entidad_id"],
    "idmunicipio": ["idmunicipio"],
    "fechahechos": ["fechahechos", "fecha_hechos", "fha_de_hchos"],
    "fechapercato": ["fechapercato", "fecha_percato"],
    "fechacaptura": ["fechacaptura", "fecha_captura"],
    "fechalocaliza": ["fechalocaliza", "fecha_localiza", "fecha_localizacion"],
    "estatus": ["estatus"],
    "delitolocalizacion": ["delitolocalizacion"],
    "expediente": ["expediente", "expedientes"],
    "dependencia_origen": ["dependencia_origen", "dependencia", "autoridad"],
    "fecha_consulta": ["fecha_consulta", "fechaconsulta"],
}
COLUMNAS_CRITICAS = ["fub", "estatus", "curp_vic"]
COLUMNAS_IDENTIDAD = [
    "curp_vic", "nombre_vic", "primerapellido_vic", "segundoapellido_vic",
    "sexo", "idestadonacimiento_vic",
]
ENTIDADES = {
    1: "AGUASCALIENTES", 2: "BAJA CALIFORNIA", 3: "BAJA CALIFORNIA SUR", 4: "CAMPECHE",
    5: "COAHUILA DE ZARAGOZA", 6: "COLIMA", 7: "CHIAPAS", 8: "CHIHUAHUA",
    9: "CIUDAD DE MEXICO", 10: "DURANGO", 11: "GUANAJUATO", 12: "GUERRERO",
    13: "HIDALGO", 14: "JALISCO", 15: "ESTADO DE MEXICO", 16: "MICHOACAN DE OCAMPO",
    17: "MORELOS", 18: "NAYARIT", 19: "NUEVO LEON", 20: "OAXACA", 21: "PUEBLA",
    22: "QUERETARO", 23: "QUINTANA ROO", 24: "SAN LUIS POTOSI", 25: "SINALOA",
    26: "SONORA", 27: "TABASCO", 28: "TAMAULIPAS", 29: "TLAXCALA", 30: "VERACRUZ",
    31: "YUCATAN", 32: "ZACATECAS", 33: "NACIDO EN EL EXTRANJERO",
}

# Diario de etapas: guarda métricas y muestras pequeñas, no DataFrames completos.
AUDITORIA = []
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Insumo local configurado: {INPUT_RNPDNO}")
print("Este cuaderno procesa archivos locales y no contiene funciones de Drive.")

## 1. Preparación del notebook y configuración inicial

La celda anterior concentra rutas, catálogos y opciones del proceso. Cambia el archivo de entrada y auxiliares en esa celda. `AUDITORIA` acumulará conteos por etapa y hoja. La lectura es local, el filtro por entidad es opcional y la exportación permanece desactivada hasta que se revise el resultado.

In [ ]:
# 2. FUNCIONES AUXILIARES: toda la lógica reutilizable vive en el cuaderno.
def normalizar_nombre_columna(nombre):
    """Convierte encabezados a minúsculas, sin acentos ni signos, con '_' entre palabras."""
    texto = unicodedata.normalize("NFKD", str(nombre))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower().strip()
    texto = re.sub(r"[\s.\-]+", "_", texto)
    return re.sub(r"[^a-z0-9_]", "", texto)


def normalizar_texto(valores):
    """Mayúsculas, sin acentos (conserva Ñ) y espacios normalizados; acepta Series o texto."""
    reemplazos = str.maketrans({"Á":"A", "É":"E", "Í":"I", "Ó":"O", "Ú":"U", "Ü":"U"})
    def uno(valor):
        if pd.isna(valor):
            return None
        return re.sub(r"\s+", " ", str(valor).upper().translate(reemplazos)).strip()
    if isinstance(valores, pd.Series):
        return valores.map(uno)
    return uno(valores)


def limpiar_curp(valores):
    """Quita espacios y pasa CURP a mayúscula; convierte cadenas vacías/NAN a nulo."""
    serie = valores if isinstance(valores, pd.Series) else pd.Series([valores])
    limpia = serie.astype("string").str.replace(r"\s+", "", regex=True).str.upper()
    limpia = limpia.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA})
    return limpia if isinstance(valores, pd.Series) else limpia.iloc[0]


def validar_curp(valores):
    """Valida estructura y clave de entidad; no comprueba el dígito verificador oficial."""
    serie = limpiar_curp(valores).astype("string")
    longitud = serie.str.len().eq(18)
    formato = serie.str.match(r"^[A-Z]{4}\d{6}[HM]", na=False)
    homoclave = serie.str.match(r"^.{13}[A-Z]{3}.{2}$", na=False)
    estado = serie.str.slice(11, 13).isin(CODIGOS_ESTADO_CURP)
    return (longitud & formato & homoclave & estado).fillna(False).astype(bool)


def detectar_formato(ruta):
    """Acepta únicamente los formatos de origen utilizados por el proceso."""
    extension = Path(ruta).suffix.lower()
    formatos = {".parquet": "parquet", ".xlsx": "excel"}
    if extension not in formatos:
        raise ValueError(f"Formato no permitido: {extension!r}. Usa .parquet o .xlsx.")
    return formatos[extension]


def convertir_fecha(serie):
    """Interpreta fechas tolerando valores inválidos; estos quedan como NaT."""
    return pd.to_datetime(serie, errors="coerce")


def vista_segura(df, n=5):
    """Muestra pocos registros y enmascara identificadores/datos personales por defecto."""
    muestra = df.head(n).copy()
    if not MOSTRAR_DATOS_SENSIBLES:
        sensibles = re.compile(
            r"fub|curp|nombre|apellido|telefono|tel_|correo|email|domicilio|direccion|"
            r"expediente|folio|reportante|contacto|referencia",
            flags=re.IGNORECASE,
        )
        for columna in muestra.columns:
            if sensibles.search(str(columna)):
                muestra[columna] = muestra[columna].map(lambda x: "[OCULTO]" if pd.notna(x) else x)
    return muestra


def auditar_dataframe(etapa, df, hoja="(todas)", mostrar=True):
    """Registra dimensiones, tipos y nulos; opcionalmente enseña una vista enmascarada."""
    nulos = df.isna().sum().sort_values(ascending=False)
    fila = {
        "etapa": etapa, "hoja": hoja, "filas": len(df), "columnas": df.shape[1],
        "memoria_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 2),
        "nulos_totales": int(df.isna().sum().sum()),
        "columnas_con_nulos": int((nulos > 0).sum()),
    }
    AUDITORIA.append(fila)
    print(f"\n[{etapa}] hoja={hoja} | {len(df):,} filas × {df.shape[1]} columnas | {fila['memoria_mb']} MB")
    print("Tipos (primeras 12 columnas):", df.dtypes.astype(str).head(12).to_dict())
    if fila["columnas_con_nulos"]:
        print("Columnas con más nulos:", nulos[nulos > 0].head(8).to_dict())
    if mostrar:
        display(vista_segura(df))
    return fila


print("Funciones auxiliares definidas: encabezados, texto, CURP, fechas y auditoría segura.")

## 2. Funciones auxiliares reutilizables

Estas funciones son definidas en el mismo notebook: normalizan nombres y texto, validan CURP, detectan extensiones, convierten fechas y crean snapshots. Las vistas ocultan identificadores por defecto; activa `MOSTRAR_DATOS_SENSIBLES` solo en un entorno protegido.

In [ ]:
# 3. CARGA LOCAL DEL INSUMO RNPDNO
# Sin Drive ni dependencias del proyecto: el flujo siempre usa esta ruta local.
if not INPUT_RNPDNO.exists():
    raise FileNotFoundError(
        f"No existe el archivo RNPDNO local: {INPUT_RNPDNO}\n"
        "Corrige INPUT_RNPDNO en la primera celda de configuración."
    )

print(f"Se leerá el archivo local: {INPUT_RNPDNO}")

## 3. Carga del insumo RNPDNO desde ruta local

Esta versión solo lee un archivo existente en el disco. No usa Drive, OAuth ni funciones importadas del proyecto. Si el archivo no existe, se detiene y muestra la ruta configurada que debe corregirse.

In [ ]:
# 4. FORMATO Y CORTE: el archivo se toma directamente de la ruta local configurada.
archivo_origen = INPUT_RNPDNO
formato_origen = detectar_formato(archivo_origen)
coincidencia_corte = re.search(r"reporte_personas_completo_(\d{2})(\d{2})(\d{4})", archivo_origen.name)
fecha_corte_nombre = None
if coincidencia_corte:
    dia, mes, anio = map(int, coincidencia_corte.groups())
    try:
        fecha_corte_nombre = date(anio, mes, dia)
    except ValueError:
        print("Aviso: el nombre tiene una fecha inválida; se determinará el corte desde fecha_consulta.")

print(f"Archivo seleccionado: {archivo_origen}")
print(f"Formato detectado: {formato_origen}")
print(f"Fecha declarada en el nombre: {fecha_corte_nombre or 'no disponible'}")
print("La fecha de corte efectiva se confirmará con fecha_consulta después de leer los datos.")

## 4. Detección del formato y fecha de corte

Se acepta `.parquet` o `.xlsx`. El nombre puede declarar una fecha en formato DDMMAAAA; después, la fecha máxima de `fecha_consulta` del contenido se usa como corte efectivo. No se infiere el corte a partir de la fecha de los hechos.

In [ ]:
# 5. LECTURA DEL ORIGEN: formar una tabla única y conservar la procedencia de cada fila.
NA_ORIGEN = ["", "#N/A", "#N/A N/A", "#NA", "<NA>", "N/A", "NA", "NULL", "NaN", "None", "nan", "null"]


def leer_origen(ruta, formato):
    """Devuelve (tabla_unificada, hoja_origen); soporta Parquet apilado y Excel de 3 hojas."""
    ruta = Path(ruta)
    if formato == "parquet":
        tabla = pd.read_parquet(ruta)
        tabla.columns = [str(c) for c in tabla.columns]
        tabla = tabla.replace(NA_ORIGEN, pd.NA).astype("string")
        requeridas = {"estatus", "conidentidad"}
        faltantes = requeridas - set(tabla.columns.str.lower())
        if faltantes:
            raise ValueError(f"Al Parquet le faltan columnas para reconstruir las hojas: {sorted(faltantes)}")
        # Hacer insensible a mayúsculas la selección de estos campos clave.
        col_estatus = next(c for c in tabla.columns if c.lower() == "estatus")
        col_identidad = next(c for c in tabla.columns if c.lower() == "conidentidad")
        estados = tabla[col_estatus].fillna("").str.strip().str.upper()
        identidad = tabla[col_identidad].fillna("").str.strip().str.lower().eq("true")
        hojas = pd.Series("PENDIENTES", index=tabla.index, dtype="string")
        hojas.loc[identidad] = "DESAPARECIDAS"
        hojas.loc[estados.isin(ESTATUS_LOCALIZADAS)] = "LOCALIZADAS"  # el estatus prevalece
    else:
        tablas, origenes = [], []
        with pd.ExcelFile(ruta) as libro:
            ausentes = [nombre for nombre in HOJAS.values() if nombre not in libro.sheet_names]
            if ausentes:
                raise ValueError(f"Faltan hojas obligatorias en el Excel: {ausentes}")
            for llave, nombre_hoja in HOJAS.items():
                print(f"Leyendo hoja {nombre_hoja}...")
                por_hoja = pd.read_excel(libro, sheet_name=nombre_hoja, dtype="string")
                tablas.append(por_hoja)
                origenes.extend([llave] * len(por_hoja))
        tabla = pd.concat(tablas, ignore_index=True, sort=False)
        tabla = tabla.replace(NA_ORIGEN, pd.NA)
        hojas = pd.Series(origenes, index=tabla.index, dtype="string")

    if not hojas.isin(HOJAS).all():
        raise ValueError(f"Hay {int((~hojas.isin(HOJAS)).sum())} filas sin hoja identificable.")
    return tabla.reset_index(drop=True), hojas.reset_index(drop=True)


rnpdno_crudo, hoja_origen = leer_origen(archivo_origen, formato_origen)
max_fecha_consulta_global = pd.NaT
if "fecha_consulta" in rnpdno_crudo.columns:
    max_fecha_consulta_global = convertir_fecha(rnpdno_crudo["fecha_consulta"]).max()
print(f"Fuente: {archivo_origen.name} | Corte efectivo (fecha_consulta máxima): {max_fecha_consulta_global}")
auditar_dataframe("01 - lectura cruda unificada", rnpdno_crudo, "todas las hojas")

## 5. Lectura del origen y tabla unificada

El lector apila las hojas del Excel y añade `hoja_origen`. El Parquet ya está apilado, por eso reconstruimos su procedencia con `estatus` y `conidentidad`; los localizados prevalecen por su estatus. Así ambos formatos producen la misma estructura de trabajo.

In [ ]:
# 6. INSPECCIÓN INICIAL: conocer tamaño, tipos, nulos y reparto por hoja/estatus.
rnpdno_crudo["hoja_origen"] = hoja_origen
print(f"Dimensiones iniciales: {rnpdno_crudo.shape[0]:,} filas × {rnpdno_crudo.shape[1]} columnas")
print("Columnas disponibles:")
display(pd.DataFrame({"columna": rnpdno_crudo.columns, "tipo": rnpdno_crudo.dtypes.astype(str).values}))
print("\nFilas por hoja de origen:")
display(hoja_origen.value_counts(dropna=False).rename_axis("hoja").to_frame("filas"))

col_estatus_inicial = next((c for c in rnpdno_crudo.columns if c.lower() == "estatus"), None)
if col_estatus_inicial:
    print("\nEstatus por hoja:")
    display(pd.crosstab(rnpdno_crudo[col_estatus_inicial], hoja_origen, dropna=False))
print("\nNulos por columna (primeras 20):")
display(rnpdno_crudo.isna().sum().sort_values(ascending=False).head(20).to_frame("nulos"))
auditar_dataframe("02 - inspección inicial", rnpdno_crudo, "todas las hojas", mostrar=False)

## 6. Inspección inicial del DataFrame

Antes de transformar se observan dimensiones, tipos, nulos, estatus y filas por hoja. La vista de ejemplo muestra campos personales enmascarados; los conteos sí reflejan el DataFrame completo.

In [ ]:
# 7-8. ESTANDARIZAR Y SEPARAR: normalizar encabezados, conservar fuente y crear dfs_por_hoja.
def estandarizar_dataframe(df, hoja):
    """Normaliza encabezados, guarda copias crudas *_rnpdno, aplica alias y revisa columnas."""
    salida = df.copy()
    salida = salida.drop(columns=["hoja_origen"], errors="ignore")
    nombres = [normalizar_nombre_columna(c) for c in salida.columns]
    repetidos = pd.Index(nombres)[pd.Index(nombres).duplicated()].unique().tolist()
    if repetidos:
        raise ValueError(f"La hoja {hoja} produce encabezados repetidos tras normalizar: {repetidos}")
    salida.columns = nombres

    # Respaldo fiel de valores de entrada ANTES de convertir tipos o renombrar alias.
    for columna in nombres:
        if not columna.endswith("_rnpdno") and f"{columna}_rnpdno" not in salida.columns:
            salida[f"{columna}_rnpdno"] = salida[columna].copy()

    # Buscar cada alias normalizado y renombrar al nombre estándar interno.
    mapa_actual = {normalizar_nombre_columna(c): c for c in salida.columns}
    cambios = {}
    for destino, alias in COLUMN_MAPPING.items():
        if destino in salida.columns:
            continue
        candidato = next((mapa_actual[normalizar_nombre_columna(a)] for a in alias
                          if normalizar_nombre_columna(a) in mapa_actual), None)
        if candidato is not None:
            cambios[candidato] = destino
    salida = salida.rename(columns=cambios)

    faltan_criticas = [c for c in COLUMNAS_CRITICAS if c not in salida.columns]
    falta_curp = "curp_vic" in faltan_criticas
    if faltan_criticas:
        print(f"AVISO {hoja}: faltan columnas críticas: {faltan_criticas}")
    if "fub" not in salida.columns or "estatus" not in salida.columns:
        raise ValueError(f"No se puede continuar con {hoja}: se requieren fub y estatus.")
    if falta_curp:
        print(f"AVISO {hoja}: no hay curp_vic; la etapa RENAPO no podrá imputar CURP.")

    # Las claves de ubicación se convierten para análisis. Sus copias *_rnpdno quedan texto.
    for columna in ["idestado", "idmunicipio", "idestadonacimiento_vic"]:
        if columna in salida.columns:
            salida[columna] = pd.to_numeric(salida[columna], errors="coerce")
    return salida


dfs_por_hoja = {}
for clave_hoja in HOJAS:
    filas_hoja = hoja_origen.eq(clave_hoja)
    df_hoja_crudo = rnpdno_crudo.loc[filas_hoja].drop(columns=["hoja_origen"]).reset_index(drop=True)
    print(f"\nHoja {clave_hoja}: {len(df_hoja_crudo):,} filas antes de estandarizar")
    df_hoja = estandarizar_dataframe(df_hoja_crudo, clave_hoja)
    dfs_por_hoja[clave_hoja] = df_hoja
    auditar_dataframe("03 - columnas estandarizadas y originales preservadas", df_hoja, clave_hoja)

print("\nClaves de dfs_por_hoja:", list(dfs_por_hoja))

## 7–8. Estandarización, separación y respaldo del origen

Los encabezados se normalizan y los alias se mapean a nombres estándar. Cada hoja se separa en su propio DataFrame; antes de convertir tipos se copian valores originales con sufijo `_rnpdno`. Las claves geográficas se convierten a número solo en las columnas de trabajo. El filtro de entidad, si se configura, no cambia la fecha global del archivo.

In [ ]:
# Filtro opcional por entidad, aplicado después de calcular el corte global.
if ID_ESTADO_FILTRO is not None:
    for hoja, df_hoja in list(dfs_por_hoja.items()):
        if "idestado" not in df_hoja.columns:
            raise KeyError(f"No se puede filtrar {hoja}: falta la columna idestado.")
        antes = len(df_hoja)
        dfs_por_hoja[hoja] = df_hoja.loc[
            pd.to_numeric(df_hoja["idestado"], errors="coerce").eq(ID_ESTADO_FILTRO)
        ].copy()
        print(f"Filtro entidad {ID_ESTADO_FILTRO} en {hoja}: {antes:,} → {len(dfs_por_hoja[hoja]):,} filas")
else:
    print("Sin filtro por entidad: se mantiene el universo nacional.")

## 9. Consistencia por FUB y deduplicación

Un FUB identifica el registro de una persona en el RNPDNO. Antes de quitar duplicados, comparamos los datos de identidad dentro de cada FUB. Si hay dos valores distintos en un campo crítico, el notebook detiene esa etapa: es preferible investigar la contradicción en vez de descartar una fila arbitrariamente. Si coinciden, conserva la primera fila.

In [ ]:
# 9. VALIDACIÓN Y DEDUPLICACIÓN: validar antes de quitar filas repetidas.
def verificar_consistencia_por_fub(df, columnas_identidad, nombre_hoja):
    """Lanza error si un FUB duplicado tiene valores distintos en columnas de identidad."""
    duplicados = df[df.duplicated("fub", keep=False)]
    if duplicados.empty:
        print(f"{nombre_hoja}: no hay FUB duplicados.")
        return
    columnas = [c for c in columnas_identidad if c in duplicados.columns]
    if not columnas:
        print(f"Aviso {nombre_hoja}: FUB duplicados, pero no hay columnas de identidad para comprobar.")
        return
    # dropna=False considera también diferencia entre faltante y valor informado.
    conteos = duplicados.groupby("fub", dropna=False)[columnas].nunique(dropna=False)
    conflictivos = conteos.gt(1).any(axis=1)
    fubs_malos = conteos.index[conflictivos].tolist()
    if fubs_malos:
        muestra_fubs = [str(x) for x in fubs_malos[:5]]
        raise ValueError(
            f"{nombre_hoja}: {len(fubs_malos)} FUB tienen contradicciones en identidad. "
            f"Ejemplos (no se muestran datos personales): cantidad={len(muestra_fubs)}. "
            "Revisa las filas repetidas antes de continuar."
        )
    print(f"{nombre_hoja}: {len(duplicados) - duplicados['fub'].nunique()} filas duplicadas consistentes.")


dfs_limpios = {}
for hoja, df in dfs_por_hoja.items():
    auditar_dataframe("antes de deduplicar", df, hoja, mostrar=False)
    verificar_consistencia_por_fub(df, COLUMNAS_IDENTIDAD, hoja)
    # Mantiene el primer registro por FUB, como en el flujo de referencia.
    df_limpio = df.drop_duplicates(subset="fub", keep="first").copy()
    dfs_limpios[hoja] = df_limpio
    print(f"{hoja}: {len(df):,} → {len(df_limpio):,} filas; se quitaron {len(df)-len(df_limpio):,}.")
    auditar_dataframe("04 - deduplicado", df_limpio, hoja)

## 10. Imputación de CURP y auditoría

Las bases RENAPO aportan CURP por FUB. Primero filtramos las CURP auxiliares que pasan las reglas de estructura. Después, solo reemplazamos la CURP del RNPDNO cuando la original no es válida y existe una válida para ese FUB. La columna original se conserva y cada reemplazo se registra en `auditoria_imputaciones`.

In [ ]:
# 10. CARGAR RENAPO, VALIDAR CURP E IMPUTAR SOLO CUANDO CORRESPONDE.
def cargar_curps_renapo(rutas):
    """Lee CSV/XLSX auxiliares, detecta columnas FUB/CURP, valida y consolida por FUB."""
    partes = []
    for ruta in rutas:
        ruta = Path(ruta)
        if not ruta.exists():
            print(f"Aviso: no se encontró archivo auxiliar (se omite): {ruta.name}")
            continue
        try:
            auxiliar = pd.read_csv(ruta, dtype="string", encoding="utf-8") if ruta.suffix.lower() == ".csv" else pd.read_excel(ruta, dtype="string")
            mapa = {normalizar_nombre_columna(c): c for c in auxiliar.columns}
            col_fub = next((v for k, v in mapa.items() if "fub" in k), None)
            col_curp = next((v for k, v in mapa.items() if "curp" in k), None)
            if col_fub is None or col_curp is None:
                print(f"Aviso: {ruta.name} no tiene columnas reconocibles de FUB y CURP; se omite.")
                continue
            parte = auxiliar[[col_fub, col_curp]].copy()
            parte.columns = ["fub", "curp_obtenida"]
            parte["fub"] = parte["fub"].astype("string").str.strip()
            parte["curp_obtenida"] = limpiar_curp(parte["curp_obtenida"])
            parte = parte[validar_curp(parte["curp_obtenida"]) & parte["fub"].notna()]
            partes.append(parte)
            print(f"{ruta.name}: {len(parte):,} pares FUB/CURP válidos.")
        except Exception as error:
            print(f"Aviso: error leyendo {ruta.name}: {error}; se omite este archivo.")
    if not partes:
        print("No se cargaron CURP auxiliares válidas; se continuará sin imputaciones.")
        return pd.DataFrame({"fub": pd.Series(dtype="string"), "curp_obtenida": pd.Series(dtype="string")})
    # Orden de RENAPO_FILES determina precedencia: ante varios valores, gana el primero.
    consolidado = pd.concat(partes, ignore_index=True).drop_duplicates("fub", keep="first")
    return consolidado.reset_index(drop=True)


def imputar_curps_por_fub(df, curps_auxiliares, hoja):
    """Añade CURP derivada, bandera de validez y registro de reemplazos."""
    # Resetear índice: el merge crea un índice nuevo, y las máscaras deben alinearse fila a fila.
    resultado = df.copy().reset_index(drop=True)
    if "curp_vic" not in resultado.columns:
        resultado["curp_vic_derivada_rnpdno"] = pd.NA
        resultado["curp_vic_valida_derivada_rnpdno"] = False
        print(f"{hoja}: falta curp_vic; todas quedan sin CURP válida.")
        return resultado, pd.DataFrame(columns=["hoja", "fub", "curp_original", "curp_nueva"])

    resultado["fub"] = resultado["fub"].astype("string").str.strip()
    resultado["curp_original_limpia"] = limpiar_curp(resultado["curp_vic"])
    valido_original = validar_curp(resultado["curp_original_limpia"])
    resultado["curp_vic_derivada_rnpdno"] = resultado["curp_original_limpia"]

    aux = curps_auxiliares.drop_duplicates("fub", keep="first").copy()
    aux["fub"] = aux["fub"].astype("string").str.strip()
    combinada = resultado.merge(aux, on="fub", how="left", validate="many_to_one")
    imputar = (~valido_original) & combinada["curp_obtenida"].notna()
    combinada.loc[imputar, "curp_vic_derivada_rnpdno"] = combinada.loc[imputar, "curp_obtenida"]
    combinada["curp_vic_valida_derivada_rnpdno"] = validar_curp(combinada["curp_vic_derivada_rnpdno"])

    cambios = combinada.loc[imputar, ["fub", "curp_original_limpia", "curp_obtenida"]].copy()
    cambios.insert(0, "hoja", hoja)
    cambios = cambios.rename(columns={"curp_original_limpia": "curp_original", "curp_obtenida": "curp_nueva"})
    combinada = combinada.drop(columns=["curp_original_limpia", "curp_obtenida"], errors="ignore")
    print(f"{hoja}: CURP válida antes={int(valido_original.sum()):,}; después={int(combinada['curp_vic_valida_derivada_rnpdno'].sum()):,}; reemplazos={len(cambios):,}.")
    return combinada, cambios


curps_renapo = cargar_curps_renapo(RENAPO_FILES)
print(f"Tabla RENAPO consolidada: {curps_renapo.shape[0]:,} filas × {curps_renapo.shape[1]} columnas")
dfs_imputados = {}
lista_cambios = []
for hoja, df in dfs_limpios.items():
    df_imp, cambios = imputar_curps_por_fub(df, curps_renapo, hoja)
    dfs_imputados[hoja] = df_imp
    lista_cambios.append(cambios)
    auditar_dataframe("05 - imputación CURP", df_imp, hoja)
auditoria_imputaciones = pd.concat(lista_cambios, ignore_index=True) if lista_cambios else pd.DataFrame()
print(f"Total de reemplazos de CURP registrados: {len(auditoria_imputaciones):,}")
# La tabla de auditoría contiene datos sensibles; no se muestra por defecto.
if MOSTRAR_DATOS_SENSIBLES and not auditoria_imputaciones.empty:
    display(auditoria_imputaciones.head(10))

## 11. Enriquecimiento de variables derivadas

En esta etapa se conservan los campos de origen y se agregan campos calculados: nombre completo normalizado, bandera de CURP, fechas convertidas, fecha de hechos de respaldo y nombre de entidad. Así se distingue el dato crudo (`*_rnpdno`) del dato preparado (`*_derivada_rnpdno`).

In [ ]:
# 11. AÑADIR CAMPOS DERIVADOS A LAS DOS VERSIONES: original limpia e imputada.
def enriquecer_dataframe(df, hoja):
    """Añade columnas derivadas sin sustituir las copias originales *_rnpdno."""
    salida = df.copy()
    componentes = []
    for columna in ["nombre_vic", "primerapellido_vic", "segundoapellido_vic"]:
        if columna in salida:
            componentes.append(salida[columna].fillna("").astype("string"))
        else:
            componentes.append(pd.Series("", index=salida.index, dtype="string"))
    nombre = (componentes[0] + " " + componentes[1] + " " + componentes[2])
    reemplazos = str.maketrans({"Á":"A", "É":"E", "Í":"I", "Ó":"O", "Ú":"U", "Ü":"U"})
    salida["nombre_completo_derivada_rnpdno"] = (
        nombre.str.upper().str.translate(reemplazos).str.replace(r"\s+", " ", regex=True).str.strip()
    )

    if "curp_vic_derivada_rnpdno" not in salida:
        salida["curp_vic_derivada_rnpdno"] = limpiar_curp(salida["curp_vic"]) if "curp_vic" in salida else pd.NA
    salida["curp_vic_valida_derivada_rnpdno"] = validar_curp(salida["curp_vic_derivada_rnpdno"])

    for columna in ["fechahechos", "fechanacimiento", "fechapercato", "fechacaptura", "fechalocaliza", "fecha_consulta"]:
        salida[f"{columna}_derivada_rnpdno"] = convertir_fecha(salida[columna]) if columna in salida else pd.NaT
    salida["fecha_hechos_modificada_derivada_rnpdno"] = (
        salida["fechahechos_derivada_rnpdno"]
        .fillna(salida["fechapercato_derivada_rnpdno"])
        .fillna(salida["fechacaptura_derivada_rnpdno"])
    )
    for columna in ["idestado", "idmunicipio"]:
        salida[f"{columna}_derivada_rnpdno"] = (
            pd.to_numeric(salida[columna], errors="coerce") if columna in salida else np.nan
        )
    salida["estado_derivada_rnpdno"] = normalizar_texto(
        salida["idestado_derivada_rnpdno"].map(ENTIDADES)
    )
    print(f"{hoja}: derivados agregados: nombre, CURP válida, fechas y ubicación.")
    return salida


# Los DataFrames imputados sirven para desaparecidas y pendientes.
dfs_imputados_enriquecidos = {
    h: enriquecer_dataframe(df, h) for h, df in dfs_imputados.items()
}
# Para localizadas se conserva el universo original sin imputar, tal como en el flujo de referencia.
dfs_limpios_enriquecidos = {
    "LOCALIZADAS": enriquecer_dataframe(dfs_limpios["LOCALIZADAS"], "LOCALIZADAS")
} if "LOCALIZADAS" in dfs_limpios else {}
for hoja, df in dfs_imputados_enriquecidos.items():
    auditar_dataframe("06 - variables derivadas", df, hoja, mostrar=(hoja == "DESAPARECIDAS"))

## 12. Segmentación: CURP válida y pendientes de actualización

El grupo **con CURP** incluye registros cuya CURP original o recuperada pasó las reglas de validación. **Pendientes** agrupa los registros no localizados que no tienen CURP válida, eliminando por FUB cualquiera que ya entró al grupo válido. Las personas localizadas se conservan como universo aparte para la consolidación final.

In [ ]:
# 12. CLASIFICAR FILAS SEGÚN LA BANDERA DE CURP; no se alteran los DataFrames de entrada.
def segmentar_bases(dfs_con_derivadas):
    """Devuelve diccionarios con_curp y sin_curp; LOCALIZADAS se reserva aparte."""
    con_curp, sin_curp = {}, {}
    fubs_con_curp = set()
    for hoja, df in dfs_con_derivadas.items():
        if "curp_vic_valida_derivada_rnpdno" not in df:
            print(f"Aviso {hoja}: falta indicador de CURP; no entra en con_curp.")
            continue
        mascara = df["curp_vic_valida_derivada_rnpdno"].fillna(False).astype(bool)
        con_curp[hoja] = df.loc[mascara].copy()
        fubs_con_curp.update(df.loc[mascara, "fub"].dropna().astype(str))

    for hoja, df in dfs_con_derivadas.items():
        if hoja == "LOCALIZADAS":
            continue
        mascara_sin = ~df["fub"].astype("string").isin(fubs_con_curp)
        sin_curp[hoja] = df.loc[mascara_sin].copy()
        if sin_curp[hoja]["curp_vic_valida_derivada_rnpdno"].fillna(False).any():
            raise AssertionError(f"La hoja {hoja} dejó CURP válida dentro de sin_curp.")
    return {"con_curp": con_curp, "sin_curp": sin_curp, "fubs_con_curp": fubs_con_curp}


bases_segmentadas = segmentar_bases(dfs_imputados_enriquecidos)
for grupo in ["con_curp", "sin_curp"]:
    print(f"\nGrupo {grupo.upper()}")
    for hoja, df in bases_segmentadas[grupo].items():
        print(f"  {hoja}: {len(df):,} filas")
        auditar_dataframe(f"07 - {grupo}", df, hoja)
print(f"FUB únicos con CURP válida: {len(bases_segmentadas['fubs_con_curp']):,}")

## 13. Consolidación del maestro en memoria

El maestro une tres universos: desaparecidas con CURP válida, pendientes sin CURP válida y todas las localizadas. Se asigna una etiqueta final y una única fecha de corte calculada con la fecha máxima de consulta del origen, antes de cualquier filtro por entidad. Se limita la salida a un esquema estable de columnas.

In [ ]:
# 13. UNIR LOS UNIVERSOS Y APLICAR EL ESQUEMA FINAL DE EXPORTACIÓN.
COLUMNAS_SALIDA = [
    "fub_rnpdno", "estatus_rnpdno", "nombre_vic_rnpdno", "primerapellido_vic_rnpdno",
    "segundoapellido_vic_rnpdno", "sexo_rnpdno", "fechanacimiento_rnpdno",
    "idestadonacimiento_vic_rnpdno", "idestado_rnpdno", "idmunicipio_rnpdno",
    "fechahechos_rnpdno", "fechapercato_rnpdno", "fechalocaliza_rnpdno",
    "delitolocalizacion_rnpdno", "fechacaptura_rnpdno", "dependencia_origen_rnpdno",
    "fecha_consulta_rnpdno", "curp_vic_rnpdno", "expedientes_rnpdno",
    "fechanacimiento_derivada_rnpdno", "idestado_derivada_rnpdno",
    "idmunicipio_derivada_rnpdno", "fechahechos_derivada_rnpdno",
    "fechapercato_derivada_rnpdno", "fechalocaliza_derivada_rnpdno",
    "fechacaptura_derivada_rnpdno", "fecha_consulta_derivada_rnpdno",
    "curp_vic_derivada_rnpdno", "curp_vic_valida_derivada_rnpdno",
    "nombre_completo_derivada_rnpdno", "fecha_hechos_modificada_derivada_rnpdno",
    "estado_derivada_rnpdno", "clasificacion_des_derivada_rnpdno",
]


def consolidar_maestro(segmentadas, originales_enriquecidas, fecha_corte):
    """Concatena desaparecidas, pendientes y localizadas, marcando clase y corte."""
    piezas = []
    desapar = [df for h, df in segmentadas["con_curp"].items() if h != "LOCALIZADAS"]
    if desapar:
        bloque = pd.concat(desapar, ignore_index=True).copy()
        bloque["clasificacion_des_derivada_rnpdno"] = "Desaparecidos"
        piezas.append(bloque)
    pendientes = list(segmentadas["sin_curp"].values())
    if pendientes:
        bloque = pd.concat(pendientes, ignore_index=True).copy()
        bloque["clasificacion_des_derivada_rnpdno"] = "Pendientes"
        piezas.append(bloque)
    if "LOCALIZADAS" in originales_enriquecidas:
        bloque = originales_enriquecidas["LOCALIZADAS"].copy()
        bloque["clasificacion_des_derivada_rnpdno"] = "Localizados"
        piezas.append(bloque)
    if not piezas:
        return pd.DataFrame(columns=COLUMNAS_SALIDA)

    maestro = pd.concat(piezas, ignore_index=True, sort=False)
    maestro["fecha_consulta_derivada_rnpdno"] = fecha_corte
    # Resolver diferencia de nomenclatura del expediente sin perder su valor crudo.
    if "expedientes_rnpdno" not in maestro and "expediente_rnpdno" in maestro:
        maestro["expedientes_rnpdno"] = maestro["expediente_rnpdno"]
    if "curp_vic_rnpdno" not in maestro and "curp_vic" in maestro:
        maestro["curp_vic_rnpdno"] = maestro["curp_vic"]
    # Mantener contrato fijo aun si una hoja no traía alguna columna opcional.
    faltan = [c for c in COLUMNAS_SALIDA if c not in maestro.columns]
    if faltan:
        print(f"Aviso: se crean como nulas columnas ausentes del origen: {faltan}")
        for columna in faltan:
            maestro[columna] = pd.NA
    maestro = maestro[COLUMNAS_SALIDA].copy()
    # Evitar tipos mixtos incompatibles al escribir Parquet.
    for columna in maestro.select_dtypes(include=["object"]).columns:
        maestro[columna] = maestro[columna].where(maestro[columna].notna(), pd.NA).astype("string")
    return maestro


fecha_corte_efectiva = max_fecha_consulta_global
if pd.isna(fecha_corte_efectiva):
    fecha_corte_efectiva = pd.Timestamp.now().normalize()
    print("Aviso: no hay fecha_consulta interpretable; se usará la fecha de hoy como corte.")
else:
    fecha_corte_efectiva = pd.Timestamp(fecha_corte_efectiva).normalize()

df_maestro = consolidar_maestro(bases_segmentadas, dfs_limpios_enriquecidos, fecha_corte_efectiva)
auditar_dataframe("08 - maestro consolidado", df_maestro, "todas las clasificaciones")

## 14. Revisión final, métricas y exportación

Comparamos tamaños por etapa, distribución final y cantidad de reemplazos. El archivo solo se escribe si `EXPORTAR_RESULTADO = True` en la configuración. Se puede elegir Parquet o CSV; conserva `df_maestro` en memoria aunque la exportación esté apagada.

In [ ]:
# 14. COMPLETAR RESPALDOS CRUDOS CON ALIAS, RECONSTRUIR EL MAESTRO Y REVISAR.
# Si un origen traía, por ejemplo, "nombre" en lugar de "nombre_vic", su copia cruda
# original tiene otro nombre. Para el esquema de salida, completar el respaldo canónico
# solo cuando falta; nunca se pisa una copia *_rnpdno ya existente.
RESPALDOS_CANONICOS = {
    "fub_rnpdno": "fub", "estatus_rnpdno": "estatus", "nombre_vic_rnpdno": "nombre_vic",
    "primerapellido_vic_rnpdno": "primerapellido_vic", "segundoapellido_vic_rnpdno": "segundoapellido_vic",
    "sexo_rnpdno": "sexo", "fechanacimiento_rnpdno": "fechanacimiento",
    "idestadonacimiento_vic_rnpdno": "idestadonacimiento_vic", "idestado_rnpdno": "idestado",
    "idmunicipio_rnpdno": "idmunicipio", "fechahechos_rnpdno": "fechahechos",
    "fechapercato_rnpdno": "fechapercato", "fechalocaliza_rnpdno": "fechalocaliza",
    "delitolocalizacion_rnpdno": "delitolocalizacion", "fechacaptura_rnpdno": "fechacaptura",
    "dependencia_origen_rnpdno": "dependencia_origen", "fecha_consulta_rnpdno": "fecha_consulta",
    "curp_vic_rnpdno": "curp_vic", "expedientes_rnpdno": "expediente",
}


def completar_respaldos(diccionario):
    """Completa copias canónicas ausentes a partir del campo estándar que no se modifica."""
    for df in diccionario.values():
        for respaldo, fuente in RESPALDOS_CANONICOS.items():
            if respaldo not in df.columns and fuente in df.columns:
                df[respaldo] = df[fuente].copy()
        # El RNPDNO a veces llama Expedientes en plural; conservar ese origen también.
        if "expedientes_rnpdno" not in df.columns and "expedientes" in df.columns:
            df["expedientes_rnpdno"] = df["expedientes"].copy()


completar_respaldos(dfs_limpios_enriquecidos)
completar_respaldos(dfs_imputados_enriquecidos)
# Recalcular la segmentación después de completar respaldos, para mantener sus copias al consolidar.
bases_segmentadas = segmentar_bases(dfs_imputados_enriquecidos)
df_maestro = consolidar_maestro(bases_segmentadas, dfs_limpios_enriquecidos, fecha_corte_efectiva)

# Resumen compacto de auditoría y comparación entre origen y maestro.
resumen_etapas = pd.DataFrame(AUDITORIA)
print("Resumen de las transformaciones ejecutadas:")
display(resumen_etapas[["etapa", "hoja", "filas", "columnas", "memoria_mb", "nulos_totales"]])
print("\nClasificación final:")
display(df_maestro["clasificacion_des_derivada_rnpdno"].value_counts(dropna=False).to_frame("registros"))
print(f"\nOrigen: {len(rnpdno_crudo):,} filas | Maestro: {len(df_maestro):,} filas | Columnas de salida: {df_maestro.shape[1]}")
print(f"Fecha de corte utilizada: {fecha_corte_efectiva}")
print(f"Reemplazos de CURP registrados: {len(auditoria_imputaciones):,}")
print("\nMuestra enmascarada del maestro:")
display(vista_segura(df_maestro))

# Exportación deliberadamente opt-in: no sobrescribe salidas hasta que se active.
FORMATO_SALIDA = "parquet"  # opciones: "parquet" o "csv"
if EXPORTAR_RESULTADO:
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    destino = OUTPUT_FILE if FORMATO_SALIDA == "parquet" else OUTPUT_FILE.with_suffix(".csv")
    if FORMATO_SALIDA == "parquet":
        df_maestro.to_parquet(destino, index=False, engine="pyarrow")
    elif FORMATO_SALIDA == "csv":
        df_maestro.to_csv(destino, index=False, encoding="utf-8-sig")
    else:
        raise ValueError("FORMATO_SALIDA debe ser 'parquet' o 'csv'.")
    print(f"Archivo exportado: {destino}")
else:
    print("Exportación omitida. Revisa métricas y activa EXPORTAR_RESULTADO cuando estés conforme.")